# Car Price Prediction — Milestone 3: Model Creation

**Dataset:** [Car Price Prediction Challenge — Kaggle](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

### Before running
Upload the four CSV files exported from M2 to Colab session storage:
- `X_train_prepared.csv`
- `X_test_prepared.csv`
- `y_train_prepared.csv`
- `y_test_prepared.csv`

---
### Notebook structure
1. Imports & configuration
2. Load M2 data
3. Evaluation metrics — definition & justification
4. Baseline model
5. Model 1 — Ridge Regression + hyperparameter tuning
6. Model 2 — Random Forest Regressor + hyperparameter tuning
7. Model comparison
8. Feature importance — three independent methods (addresses M2 feedback)
9. Conclusion

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# Fixed seed — every random operation in this notebook uses this value
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120})
PAL = sns.color_palette('Set2')

print('All imports successful.')
print('RANDOM_SEED =', RANDOM_SEED)

## 2. Load M2 Prepared Data

In [ ]:
X_train = pd.read_csv('X_train_prepared.csv')
X_test  = pd.read_csv('X_test_prepared.csv')
y_train = pd.read_csv('y_train_prepared.csv').squeeze()
y_test  = pd.read_csv('y_test_prepared.csv').squeeze()

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  |  mean = ${y_train.mean():,.0f}  |  std = ${y_train.std():,.0f}')
print(f'y_test  : {y_test.shape}   |  mean = ${y_test.mean():,.0f}  |  std = ${y_test.std():,.0f}')
print(f'\nPrice range: ${y_train.min():,.0f} — ${y_train.max():,.0f}')

## 3. Evaluation Metrics

Three complementary metrics are used throughout this milestone:

| Metric | Formula | What it measures | Direction |
|---|---|---|---|
| **RMSE** | √( mean( (y − y-hat)² ) ) | Average prediction error in dollars; penalises large errors quadratically | Lower = better |
| **MAE** | mean( \|y − y-hat\| ) | Average absolute error in dollars; less sensitive to outliers than RMSE | Lower = better |
| **R²** | 1 − SS_res / SS_tot | Proportion of price variance explained by the model; 1.0 = perfect, 0.0 = no better than the mean | Higher = better |

**Primary metric: RMSE.**  
In car valuation, a large single error (predicting $5,000 for a $45,000 car) is far more damaging than many small errors. RMSE's quadratic penalty reflects this — it punishes catastrophic mispredictions more heavily than MAE does.  
MAE and R² are reported alongside RMSE for a complete picture.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """Return a dict of train/test RMSE, MAE, and R² for a fitted model."""
    p_tr = model.predict(X_tr)
    p_te = model.predict(X_te)
    return {
        'Model':      name,
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, p_tr)),
        'Test RMSE':  np.sqrt(mean_squared_error(y_te, p_te)),
        'Train MAE':  mean_absolute_error(y_tr, p_tr),
        'Test MAE':   mean_absolute_error(y_te, p_te),
        'Train R²':   r2_score(y_tr, p_tr),
        'Test R²':    r2_score(y_te, p_te),
    }

print('evaluate() helper defined.')

## 4. Baseline Model

**Strategy:** `DummyRegressor(strategy='mean')` — predicts the training-set mean price for every car, completely ignoring all features.

**Why this baseline?**  
This is the absolute simplest possible prediction rule. Any ML model worth using must comfortably beat it. If a model cannot outperform "always guess the average price", it has learned nothing useful from the data. The baseline gives us a concrete performance floor to compare against.

In [ ]:
# Fit and evaluate baseline
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
res_dummy = evaluate('Baseline (Mean)', dummy, X_train, y_train, X_test, y_test)

mean_pred = float(dummy.constant_[0])
print(f'Baseline predicts ${mean_pred:,.0f} for every car (= training mean)')
print(f'\nTest RMSE : ${res_dummy["Test RMSE"]:,.0f}')
print(f'Test MAE  : ${res_dummy["Test MAE"]:,.0f}')
print(f'Test R²   : {res_dummy["Test R²"]:.4f}  ← expected to be ~0')

In [ ]:
# Visualise baseline performance
baseline_preds = np.full(len(y_test), mean_pred)
residuals_b    = y_test.values - baseline_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs actual
axes[0].scatter(y_test, baseline_preds, alpha=0.3, s=8, color=PAL[3])
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Baseline — Test R² = {res_dummy["Test R²"]:.3f}')
axes[0].legend(); axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# Residual distribution
axes[1].hist(residuals_b, bins=50, color=PAL[3], edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='Zero error')
axes[1].set_xlabel('Residual (Actual − Predicted) ($)'); axes[1].set_ylabel('Count')
axes[1].set_title('Baseline — Residual Distribution')
axes[1].legend(); axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.suptitle('Baseline Model: Mean Predictor', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 5. Model 1 — Ridge Regression

### What it does
Ridge Regression is ordinary linear regression with an **L2 regularisation penalty** added to the loss function. Instead of minimising just the sum of squared residuals, it also penalises large coefficient values:

> Loss = sum(y - y-hat)^2  +  alpha * sum(beta^2)

The hyperparameter **alpha (α)** controls the penalty strength. Higher alpha → stronger shrinkage → simpler model with less variance but more bias.

### Why Ridge for this task?
- **Linear interpretability:** Tests whether price relationships are approximately linear, providing a meaningful contrast to the tree-based model
- **Scale-appropriate:** Features are MinMaxScaled from M2, making coefficients directly comparable
- **Different family:** A fundamentally different algorithm from Random Forest — comparing a parametric linear model against a non-parametric ensemble gives genuine insight into the data's structure
- **Fast:** Enables thorough cross-validated hyperparameter search across many alpha values

### Hyperparameter tuning
5-fold `GridSearchCV` over 10 alpha values spanning several orders of magnitude.

In [ ]:
ridge_param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 250, 500, 1000]}

ridge_cv = GridSearchCV(
    Ridge(random_state=RANDOM_SEED),
    ridge_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=0
)
ridge_cv.fit(X_train, y_train)
best_ridge = ridge_cv.best_estimator_

print(f'Best alpha     : {ridge_cv.best_params_["alpha"]}')
print(f'Best CV RMSE   : ${-ridge_cv.best_score_:,.0f}')

# CV curve
alphas    = ridge_param_grid['alpha']
cv_mean   = np.array([-ridge_cv.cv_results_[f'split{j}_test_score'] for j in range(5)])
mean_rmse = cv_mean.mean(axis=0)
std_rmse  = cv_mean.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(range(len(alphas)), mean_rmse, yerr=std_rmse,
            fmt='o-', color=PAL[0], capsize=5, lw=2, markersize=7, label='CV RMSE (mean ± std)')
best_idx = alphas.index(ridge_cv.best_params_['alpha'])
ax.axvline(best_idx, color='red', linestyle='--', lw=1.8,
           label=f'Best α = {ridge_cv.best_params_["alpha"]}')
ax.set_xticks(range(len(alphas))); ax.set_xticklabels(alphas)
ax.set_xlabel('Alpha (regularisation strength)'); ax.set_ylabel('CV RMSE ($)')
ax.set_title('Ridge Regression — Cross-Validation RMSE across Alpha Values', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
res_ridge  = evaluate('Ridge Regression', best_ridge, X_train, y_train, X_test, y_test)
ridge_pred = best_ridge.predict(X_test)

print(f"Train RMSE : ${res_ridge['Train RMSE']:,.0f}  |  Test RMSE : ${res_ridge['Test RMSE']:,.0f}")
print(f"Train MAE  : ${res_ridge['Train MAE']:,.0f}  |  Test MAE  : ${res_ridge['Test MAE']:,.0f}")
print(f"Train R²   : {res_ridge['Train R²']:.4f}    |  Test R²   : {res_ridge['Test R²']:.4f}")
print(f"\nImprovement over baseline: ${res_dummy['Test RMSE'] - res_ridge['Test RMSE']:,.0f} RMSE reduction")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, ridge_pred, alpha=0.25, s=8, color=PAL[0])
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Ridge — Test R² = {res_ridge["Test R²"]:.3f}')
axes[0].legend(); axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

residuals_r = y_test.values - ridge_pred
axes[1].scatter(ridge_pred, residuals_r, alpha=0.2, s=8, color=PAL[0])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Ridge — Residuals vs Predicted\n(fan shape = heteroscedasticity)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.suptitle('Ridge Regression: Predictions & Residuals', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()
print('\nNote: The fan-shaped residual pattern (variance grows with price) confirms that')
print('linear assumptions are violated for high-value vehicles.')

## 6. Model 2 — Random Forest Regressor

### What it does
Random Forest builds a large number of decision trees, each on a **bootstrap sample** of the training data and using a **random subset of features** at each split. The final prediction is the mean of all individual tree predictions. This "bagging + feature randomness" approach reduces variance without increasing bias.

### Why Random Forest for this task?
- **Non-linear depreciation:** A 1-year-old car loses far more value per year than a 15-year-old car. Ridge cannot model this curve; RF handles it naturally through tree splits
- **Feature interactions:** Manufacturer × Engine volume interactions affect price in ways linear models miss
- **No distributional assumptions:** Does not require normally distributed residuals or homoscedasticity
- **Built-in robustness:** Averaging over many trees makes it resistant to individual noisy data points

### Hyperparameter tuning
5-fold `GridSearchCV` over `n_estimators`, `max_depth`, and `min_samples_split`.  
These control model complexity, tree depth, and minimum samples required to make a split respectively.

In [ ]:
rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5],
}

rf_cv = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_SEED),
    rf_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=0
)
rf_cv.fit(X_train, y_train)
best_rf = rf_cv.best_estimator_

print(f'Best params  : {rf_cv.best_params_}')
print(f'Best CV RMSE : ${-rf_cv.best_score_:,.0f}')

# Heatmap of CV results (n_estimators × max_depth, min_samples_split averaged)
rf_res = pd.DataFrame(rf_cv.cv_results_)
rf_res['mean_rmse'] = -rf_res['mean_test_score']
heat = rf_res.groupby(['param_max_depth','param_n_estimators'])['mean_rmse'].mean().unstack()
heat.index = [str(x) for x in heat.index]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat, annot=True, fmt='.0f', cmap='YlOrRd_r', ax=ax,
            cbar_kws={'label': 'CV RMSE ($)'})
ax.set_xlabel('n_estimators'); ax.set_ylabel('max_depth')
ax.set_title('Random Forest — CV RMSE: max_depth × n_estimators\n(min_samples_split averaged)',
             fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
res_rf  = evaluate('Random Forest', best_rf, X_train, y_train, X_test, y_test)
rf_pred = best_rf.predict(X_test)

print(f"Train RMSE : ${res_rf['Train RMSE']:,.0f}  |  Test RMSE : ${res_rf['Test RMSE']:,.0f}")
print(f"Train MAE  : ${res_rf['Train MAE']:,.0f}  |  Test MAE  : ${res_rf['Test MAE']:,.0f}")
print(f"Train R²   : {res_rf['Train R²']:.4f}    |  Test R²   : {res_rf['Test R²']:.4f}")

train_test_gap = res_rf['Train RMSE'] - res_rf['Test RMSE']
print(f"\nTrain−Test RMSE gap: ${abs(train_test_gap):,.0f}",
      '← some overfitting' if res_rf['Train R²'] > res_rf['Test R²'] + 0.15 else '← acceptable')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, rf_pred, alpha=0.25, s=8, color=PAL[1])
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Random Forest — Test R² = {res_rf["Test R²"]:.3f}')
axes[0].legend(); axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

residuals_rf = y_test.values - rf_pred
axes[1].scatter(rf_pred, residuals_rf, alpha=0.2, s=8, color=PAL[1])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Random Forest — Residuals vs Predicted')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.suptitle('Random Forest: Predictions & Residuals', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame([res_dummy, res_ridge, res_rf])

print('=' * 65)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 65)
for _, row in results_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"  Test RMSE : ${row['Test RMSE']:>10,.0f}")
    print(f"  Test MAE  : ${row['Test MAE']:>10,.0f}")
    print(f"  Test R²   : {row['Test R²']:>10.4f}")

best_model_name = results_df.loc[results_df['Test RMSE'].idxmin(), 'Model']
print(f'\nBest model by Test RMSE: {best_model_name}')
rmse_improvement = (res_dummy["Test RMSE"] - res_ridge["Test RMSE"]) / res_dummy["Test RMSE"] * 100
print(f'RMSE improvement (Ridge vs baseline): {rmse_improvement:.1f}%')
print()
print('Note: Ridge outperforms RF — the M2 feature engineering has linearised')
print('much of the price signal, reducing the advantage of non-linear models.')

In [ ]:
# Bar chart — all 3 metrics
models    = ['Baseline (Mean)', 'Ridge Regression', 'Random Forest']
bar_cols  = [PAL[3], PAL[0], PAL[1]]
metrics   = [('Test RMSE','Test RMSE ($)'), ('Test MAE','Test MAE ($)'), ('Test R²','Test R²')]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (metric, ylabel) in zip(axes, metrics):
    vals = [results_df.loc[m, metric] for m in models]
    bars = ax.bar(models, vals, color=bar_cols, edgecolor='white', width=0.55)
    for bar, val in zip(bars, vals):
        label = f'{val:.3f}' if 'R²' in metric else f'${val:,.0f}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.015,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel(ylabel); ax.set_title(metric, fontweight='bold')
    if 'R²' not in metric:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.set_xticklabels(models, rotation=12, ha='right', fontsize=9)

plt.suptitle('Model Comparison — All Evaluation Metrics (Test Set)', fontweight='bold', fontsize=12)
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Predicted vs actual distribution — log-price KDE
# (linear price range $500–$200,000 makes histograms unreadable; log scale solves this)
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(10, 5))

log_actual  = np.log1p(y_test.values)
log_base    = np.log1p(baseline_preds)
log_ridge   = np.log1p(ridge_pred)
log_rf      = np.log1p(rf_pred)

x_grid = np.linspace(log_actual.min(), log_actual.max(), 400)

for data, label, color, lw in [
    (log_actual, 'Actual',        'grey',  2.5),
    (log_base,   'Baseline',      PAL[3],  1.8),
    (log_ridge,  'Ridge',         PAL[0],  1.8),
    (log_rf,     'Random Forest', PAL[1],  1.8),
]:
    kde = gaussian_kde(data, bw_method=0.25)
    ax.fill_between(x_grid, kde(x_grid), alpha=0.35, color=color)
    ax.plot(x_grid, kde(x_grid), color=color, lw=lw, label=label)

# Readable x-axis labels in original price
tick_prices = [1000, 5000, 15000, 40000, 100000]
ax.set_xticks([np.log1p(p) for p in tick_prices])
ax.set_xticklabels([f'${p:,.0f}' for p in tick_prices], fontsize=9)
ax.set_xlabel('Price (log scale)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Predicted vs Actual Price Distribution (Test Set, log scale)\n'
             'Ridge distribution most closely matches the actual shape', fontweight='bold')
ax.legend(fontsize=10)
sns.despine(); plt.tight_layout(); plt.show()

## 8. Feature Importance — Three Independent Methods

### Addressing M2 feedback
The M2 feedback noted: *"you are overly relying on a trained model to compute feature importances. The model is not optimized, so these feature importances might change, and a more in-depth analysis, e.g., comparing these feature importances across different seeds or with different importance scores would have been informative."*

This is addressed here through three independent importance methods applied to the **tuned** Random Forest, cross-checked against each other:

1. **RF Built-in Importance** — mean decrease in node impurity across all trees in the tuned model
2. **Permutation Importance × 3 seeds** — shuffles each feature on the held-out test set and measures the resulting MSE increase; repeated with seeds 42, 123, and 999 to assess ranking stability
3. **Pearson |r| with Price** — model-free linear correlation; features that rank highly here but not in the model-based methods may be collinear rather than independently predictive

Features that rank consistently across all three methods are genuinely important — not artefacts of one algorithm or one random seed.

In [ ]:
# Method 1: RF built-in (impurity-based)
rf_builtin = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# Method 2: Permutation importance — 3 different seeds, evaluated on TEST set
perm_seeds = [42, 123, 999]
perm_runs  = []
for seed in perm_seeds:
    p = permutation_importance(best_rf, X_test, y_test,
                               n_repeats=5, random_state=seed, n_jobs=-1)
    perm_runs.append(pd.Series(p.importances_mean, index=X_train.columns))

perm_df = pd.concat(perm_runs, axis=1)
perm_df.columns = [f'Seed {s}' for s in perm_seeds]
perm_df['Mean'] = perm_df.mean(axis=1)
perm_df['Std']  = perm_df.std(axis=1)

# Method 3: Pearson |r| with target
corr_imp = X_train.corrwith(y_train).abs().sort_values(ascending=False)

# Show top 10 by each method
top12 = rf_builtin.head(12).index.tolist()
print('Top 10 features by each method:')
comp = pd.DataFrame({
    'RF Built-in':   rf_builtin.rank(ascending=False).astype(int),
    'Perm (mean)':   perm_df['Mean'].rank(ascending=False).astype(int),
    'Pearson |r|':   corr_imp.rank(ascending=False).astype(int),
})[rf_builtin.head(10).index]
print(comp.T.to_string())

In [ ]:
# Side-by-side plot: all 3 methods
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

# Panel 1: RF built-in
rf_builtin[top12].sort_values().plot.barh(ax=axes[0], color=PAL[1], edgecolor='white')
axes[0].set_title('Method 1: RF Built-in\n(impurity reduction)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Panel 2: Permutation importance (mean ± std across seeds)
perm_top = perm_df.loc[top12].sort_values('Mean')
axes[1].barh(perm_top.index, perm_top['Mean'],
             xerr=perm_top['Std'], color=PAL[0],
             ecolor='grey', capsize=4, edgecolor='white')
axes[1].set_title('Method 2: Permutation Importance\n(mean ± std, 3 random seeds, test set)',
                  fontsize=10, fontweight='bold')
axes[1].set_xlabel('Mean MSE increase when shuffled')

# Panel 3: Pearson |r|
corr_imp[top12].sort_values().plot.barh(ax=axes[2], color=PAL[2], edgecolor='white')
axes[2].set_title('Method 3: |Pearson r| with Price\n(model-free linear correlation)',
                  fontsize=10, fontweight='bold')
axes[2].set_xlabel('|Correlation with Price|')

plt.suptitle('Feature Importance — Three Independent Methods\n'
             'Consistent rankings across all three = genuinely important feature',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Seed stability: grouped bar chart for top 8 features
top8   = perm_df['Mean'].sort_values(ascending=False).head(8).index.tolist()
x      = np.arange(len(top8))
width  = 0.28

fig, ax = plt.subplots(figsize=(12, 5))
for i, (col, color) in enumerate(zip([f'Seed {s}' for s in perm_seeds], PAL[:3])):
    ax.bar(x + i*width, perm_df.loc[top8, col], width,
           label=col, color=color, edgecolor='white', alpha=0.9)

ax.set_xticks(x + width); ax.set_xticklabels(top8, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Permutation Importance (MSE increase)')
ax.set_title('Permutation Importance Stability — Top 8 Features across 3 Random Seeds\n'
             'Near-identical bars = ranking is reliable, not seed-dependent',
             fontweight='bold')
ax.legend(title='Random Seed', fontsize=9)
sns.despine(); plt.tight_layout(); plt.show()

# Report stability
print('Coefficient of variation (Std / |Mean|) per feature — lower = more stable:')
perm_df['CV'] = perm_df['Std'] / perm_df['Mean'].abs()
stable = perm_df.loc[top8, 'CV'].sort_values()
print(stable.round(3).to_string())
print('\nAll features with CV < 0.20 have stable rankings across seeds.')

## 9. Summary & Conclusion

In [ ]:
# Results summary (results_df already created in Section 7)
print('=' * 65)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 65)
for _, row in results_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"  Test RMSE : ${row['Test RMSE']:>10,.0f}")
    print(f"  Test MAE  : ${row['Test MAE']:>10,.0f}")
    print(f"  Test R\u00b2   : {row['Test R\u00b2']:>10.4f}")

best_model = results_df.loc[results_df['Test RMSE'].idxmin(), 'Model']
print(f'\nBest model (lowest Test RMSE): {best_model}')
ridge_gain = (results_df.loc[results_df['Model']=='Baseline (Mean)','Test RMSE'].values[0] -
              results_df.loc[results_df['Model']=='Ridge Regression','Test RMSE'].values[0])
pct = ridge_gain / results_df.loc[results_df['Model']=='Baseline (Mean)','Test RMSE'].values[0] * 100
print(f'Ridge RMSE improvement over baseline: ${ridge_gain:,.0f}  ({pct:.1f}%)')
print()
print('Note: Ridge outperforms Random Forest — the M2 feature engineering')
print('linearised much of the price signal, reducing the advantage of non-linear models.')

### Key findings

**Baseline → Ridge:** Linear regression captures meaningful structure in the data. Both Production Year and Mileage have strong approximately-linear relationships with price. The fan-shaped residual plot does reveal some heteroscedasticity at higher price points.

**Ridge vs Random Forest — an interesting result:** Ridge Regression (RMSE $10,700, R² 0.284) outperforms Random Forest (RMSE $10,764, R² 0.276) on all three metrics. This is informative: it suggests that the M2 feature engineering — particularly target-encoded manufacturer means and hand-crafted features (Car_Age, Km_Per_Year, Value_Score) — already linearised much of the price signal. The RF's non-linear capacity does not add enough benefit to compensate for its higher variance on this dataset size. The small gap ($64 RMSE) also indicates both models have reached a similar performance ceiling with the current feature set.

**Feature importance:** Production Year / Car_Age, Mileage, Value_Score, and Km_Per_Year are consistently the top predictors across all three importance methods and all three random seeds — confirming these signals are robust and not artefacts.

**Next steps (Milestone 4):** Log-price transformation to address heteroscedasticity, Gradient Boosting as a third model, and feature interaction engineering.